In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import label
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [2]:
import os
try:
    os.chdir("/home/maniacalm/bin/VQVAE_studies")
except FileNotFoundError:
    print("Either filepath is wrong or this is ran on another computer")

In [3]:
import wandb

#dataloader
target_snr, pulse_length = 0.2,512
training_length,test_length,val_length = 10000,1000,1000
training_seed,test_seed,val_seed = 0,training_length*2,training_length*3
batch_size = 64

#model
in_ch=1
hid=32
z_ch=512
n_codes=32

#adam
lr=1e-4
weight_decay=1e-4

#lr_scheduler
mode="min"
factor=0.5
patience=10
min_lr=1e-5

#training
num_epochs = 100


# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="gwyndandy-niu",
    # Set the wandb project where this run will be logged.
    project="VQ-VAE Init Testing",
    # Track hyperparameters and run metadata.
    config={
        #dataloader
        'target_snr':target_snr,
        'pulse_length':pulse_length,
        'training_length':training_length,
        'test_length':test_length,
        'val_length':val_length,
        'training_seed':training_seed,
        'test_seed':test_seed,
        'val_seed':val_seed,
        'batch_size':batch_size,

        #model
        'in_ch':in_ch,
        'hid':hid,
        'z_ch':z_ch,
        'n_codes':n_codes,

        #adam
        'lr':lr,
        'weight_decay':weight_decay,

        #lr_scheduler
        'mode':mode,
        'factor':factor,
        'patience':patience,
        'min_lr':min_lr,

        #training
        'num_epochs':num_epochs,
    },
)

wandb: Currently logged in as: gwyndandy (gwyndandy-niu). Use `wandb login --relogin` to force relogin


In [4]:
import numpy as np
import matplotlib.pyplot as plt

a=np.load("snb-Z-signal-0.npy",allow_pickle=True)
c=np.load("snb-Z-clnsig-0.npy",allow_pickle=True)
nwf=a.size
ntcks=len(a[0])-54
train_noise = []
train_signal = []
for j in range(nwf-50):
    ya=[]
    yc=[]
    j=0
    for i in range(0,ntcks):
        lab='tck_'+str(i)
        ya.append(a[j][lab])
        yc.append(c[j][lab])
    train_noise.append(ya)
    train_signal.append(yc)
test_noise = []
test_signal = []
for j in range(nwf-50,nwf-1):
    ya=[]
    yc=[]
    j=0
    for i in range(0,ntcks):
        lab='tck_'+str(i)
        ya.append(a[j][lab])
        yc.append(c[j][lab])
    test_noise.append(ya)
    test_signal.append(yc)
train_noise_c = np.reshape(np.array(train_noise), (-1, 1,512))
train_signal_c = np.reshape(np.array(train_signal), (-1, 1,512))
test_noise_c = np.reshape(np.array(test_noise), (-1, 1,512))
test_signal_c = np.reshape(np.array(test_signal), (-1, 1,512))
print(len(train_noise_c))
print(train_noise_c.shape)

196
(196, 1, 512)


In [5]:
class NoisyToClean1DDataset(Dataset):
    def __init__(self, noisy, clean, normalize=True):
        self.noisy = noisy.astype(np.float32)
        self.clean = clean.astype(np.float32)
        self.normalize = normalize

    def __len__(self):
        return len(self.noisy)

    def __getitem__(self, idx):
        x = self.noisy[idx]   # (1, 512)
        y = self.clean[idx]   # (1, 512)

        if self.normalize:
            mu = x.mean()
            sd = x.std() + 1e-6
            x = (x - mu) / sd
            y = (y - mu) / sd

        return {
            "x": torch.from_numpy(x).float(),
            "y": torch.from_numpy(y).float(),
        }

In [6]:
train_ds_full = NoisyToClean1DDataset(
    noisy=train_noise_c,
    clean=train_signal_c,
    normalize=True,
)

test_ds = NoisyToClean1DDataset(
    noisy=test_noise_c,
    clean=test_signal_c,
    normalize=True,
)

n_val = int(0.1 * len(train_ds_full))
n_train = len(train_ds_full) - n_val

train_ds, val_ds = random_split(
    train_ds_full,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42),
)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

batch = next(iter(train_loader))
batch["x"].shape, batch["y"].shape

(torch.Size([64, 1, 512]), torch.Size([64, 1, 512]))

In [7]:
import numpy.random as rng

def get_gaussian(mu,sigma,pulse_length):
    x = np.arange(pulse_length)
    a = -0.5 * (x - mu)**2 / sigma**2
    return np.exp(a) / (sigma * np.sqrt(2 * np.pi))

class simulacra_dataset(Dataset):
    """simulacra of simulated data of neutrinos dataset."""

    def __init__(self, target_snr, length, seed, pulse_length):
        """
        Arguments:
            csv_file (string): Path to the csv file with annotations.
            root_dir (string): Directory with all the images.
            transform (callable, optional): Optional transform to be applied
                on a sample.
        """
        
        self.target_snr = target_snr
        self.length = length
        self.seed = seed
        self.pulse_length = pulse_length
        self.min_mu = int(pulse_length*0.1)
        self.max_mu = int(pulse_length*0.9)
        self.min_sigma = int(pulse_length*0.05)
        self.max_sigma = int(pulse_length*0.1)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        rng.seed(idx+self.seed)
        num_pulse = rng.randint(1,3)
        bool_inverse = [rng.randint(0, 100) > 50 for _ in range(num_pulse)]

        clean = np.zeros([self.pulse_length])
        for inverse in bool_inverse:
            mu = rng.randint(self.min_mu,self.max_mu)
            sigma = rng.randint(self.min_sigma,self.max_sigma)
            clean += get_gaussian(mu,sigma,self.pulse_length)

            if inverse:
                weight = -rng.random()
                clean += get_gaussian(mu+sigma,sigma,self.pulse_length)*weight

        mu, sigma = 0, 0.32
        noise = np.random.normal(mu, sigma, self.pulse_length)
        
        signal_rms = np.sqrt(np.mean(clean**2))
        noise_rms = np.sqrt(np.mean(noise**2))
        
        weight = self.target_snr * noise_rms / signal_rms

        clean *= weight
        result = clean + noise
        normalization_constant =  max(max(result),max(clean))
        return {
            "x": torch.from_numpy(result/normalization_constant).float().unsqueeze(0),
            "y": torch.from_numpy(clean/normalization_constant).float().unsqueeze(0),
        }

train_ds = simulacra_dataset(target_snr, training_length, training_seed, pulse_length)
val_ds = simulacra_dataset(target_snr, val_length, val_seed, pulse_length)
test_ds = simulacra_dataset(target_snr, test_length, test_seed, pulse_length)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

batch = next(iter(train_loader))
batch["x"].shape, batch["y"].shape

(torch.Size([64, 1, 512]), torch.Size([64, 1, 512]))

In [8]:
class ResBlock1D(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(c, c, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(c, c, 3, padding=1),
        )

    def forward(self, x):
        return F.relu(x + self.net(x))


class Encoder1D(nn.Module):
    def __init__(self, in_ch=1, hid=64, z_ch=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, hid, 4, stride=2, padding=1),  # 512 -> 256
            nn.Tanh(),

            nn.Conv1d(hid, hid, 4, stride=2, padding=1),    # 256 -> 128
            nn.Tanh(),

            ResBlock1D(hid),

            nn.Conv1d(hid, z_ch, 1),
        )

    def forward(self, x):
        return self.net(x)


class Decoder1D(nn.Module):
    def __init__(self, out_ch=1, hid=64, z_ch=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(z_ch, hid, 1),

            ResBlock1D(hid),

            nn.ConvTranspose1d(hid, hid, 4, stride=2, padding=1),  # 128 -> 256
            nn.Tanh(),

            nn.ConvTranspose1d(hid, hid, 4, stride=2, padding=1),  # 256 -> 512
            nn.Tanh(),

            nn.Conv1d(hid, out_ch, 3, padding=1),
        )
    def forward(self, z):
        return self.net(z)


class VectorQuantizer1D(nn.Module):
    def __init__(self, n_codes=512, code_dim=64, beta=0.25):
        super().__init__()
        self.beta = beta
        self.codebook = nn.Embedding(n_codes, code_dim)
        self.codebook.weight.data.uniform_(-1 / n_codes, 1 / n_codes)

    def forward(self, z_e):
        # z_e: (B, C, L)
        B, C, L = z_e.shape

        z = z_e.permute(0, 2, 1).contiguous().view(-1, C)

        d = (
            z.pow(2).sum(1, keepdim=True)
            + self.codebook.weight.pow(2).sum(1)
            - 2 * z @ self.codebook.weight.t()
        )

        idx = torch.argmin(d, dim=1)

        z_q = self.codebook(idx)
        z_q = z_q.view(B, L, C).permute(0, 2, 1).contiguous()

        codebook_loss = F.mse_loss(z_q, z_e.detach(),reduction="mean")
        commit_loss = F.mse_loss(z_e, z_q.detach(),reduction="mean")
        vq_loss = codebook_loss + self.beta * commit_loss

        z_q = z_e + (z_q - z_e).detach()

        return z_q, vq_loss, idx.view(B, L)


class VQVAE1DDenoiser(nn.Module):
    def __init__(self, in_ch=1, hid=64, z_ch=64, n_codes=512):
        super().__init__()
        self.enc = Encoder1D(in_ch, hid, z_ch)
        self.vq = VectorQuantizer1D(n_codes, z_ch)
        self.dec = Decoder1D(1, hid, z_ch)

    def init_weights(m):
        if isinstance(m, (nn.Conv1d, nn.ConvTranspose1d)):
            nn.init.kaiming_uniform_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, x):
        z_e = self.enc(x)
        z_q, vq_loss, codes = self.vq(z_e)
        clean_hat = self.dec(z_q)
        return clean_hat, vq_loss, codes

In [9]:
model = VQVAE1DDenoiser(in_ch, hid, z_ch, n_codes).to(device)

model

VQVAE1DDenoiser(
  (enc): Encoder1D(
    (net): Sequential(
      (0): Conv1d(1, 32, kernel_size=(4,), stride=(2,), padding=(1,))
      (1): Tanh()
      (2): Conv1d(32, 32, kernel_size=(4,), stride=(2,), padding=(1,))
      (3): Tanh()
      (4): ResBlock1D(
        (net): Sequential(
          (0): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
          (1): ReLU(inplace=True)
          (2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
        )
      )
      (5): Conv1d(32, 512, kernel_size=(1,), stride=(1,))
    )
  )
  (vq): VectorQuantizer1D(
    (codebook): Embedding(32, 512)
  )
  (dec): Decoder1D(
    (net): Sequential(
      (0): Conv1d(512, 32, kernel_size=(1,), stride=(1,))
      (1): ResBlock1D(
        (net): Sequential(
          (0): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
          (1): ReLU(inplace=True)
          (2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
        )
      )
      (2): ConvTranspose

In [10]:
batch = next(iter(train_loader))

x = batch["x"].to(device)
y = batch["y"].to(device)

with torch.no_grad():
    y_hat, vq_loss, codes = model(x)

print("x:", x.shape)
print("y:", y.shape)
print("y_hat:", y_hat.shape)
print("codes:", codes.shape)
print("vq_loss:", vq_loss.item())

x: torch.Size([64, 1, 512])
y: torch.Size([64, 1, 512])
y_hat: torch.Size([64, 1, 512])
codes: torch.Size([64, 128])
vq_loss: 0.018521495163440704


In [11]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=lr,
    weight_decay=weight_decay
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=mode,
    factor=factor,
    patience=patience,
    min_lr=min_lr,
)

In [12]:
def run_epoch(loader, train=True, lambda_vq=0.25, max_grad_norm=1.0):
    model.train() if train else model.eval()

    total_loss = 0.0
    total_recon = 0.0
    total_vq = 0.0
    skipped = 0

    for batch in loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)

        with torch.set_grad_enabled(train):
            y_hat, vq_loss, codes = model(x)

            recon_loss = F.huber_loss(y_hat, y, delta=0.5)
            loss =  recon_loss + lambda_vq * vq_loss

            if not torch.isfinite(loss):
                skipped += 1
                continue

            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=max_grad_norm,
                )

                optimizer.step()

        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_vq += vq_loss.item()

    n = max(1, len(loader) - skipped)

    return {
        "loss": total_loss / n,
        "recon": total_recon / n,
        "vq": total_vq / n,
        "skipped": skipped,
    }

In [13]:
history = {
    "train_loss": [],
    "val_loss": [],
    "train_recon": [],
    "val_recon": [],
    "train_vq": [],
    "val_vq": [],
    "lr": [],
}

best_val = float("inf")
best_state = None
patience = 15
bad_epochs = 5

for epoch in range(1, num_epochs + 1):

    # Warm up VQ loss so it does not dominate early training
    lambda_vq = min(0.15, 0.15 * epoch / 10)

    train_stats = run_epoch(
        train_loader,
        train=True,
        lambda_vq=lambda_vq,
        max_grad_norm=1.0,
    )

    val_stats = run_epoch(
        val_loader,
        train=False,
        lambda_vq=lambda_vq,
        max_grad_norm=1.0,
    )

    scheduler.step(val_stats["loss"])

    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_stats["loss"])
    history["val_loss"].append(val_stats["loss"])
    history["train_recon"].append(train_stats["recon"])
    history["val_recon"].append(val_stats["recon"])
    history["train_vq"].append(train_stats["vq"])
    history["val_vq"].append(val_stats["vq"])
    history["lr"].append(current_lr)

    if val_stats["loss"] < best_val:
        best_val = val_stats["loss"]
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }
        bad_epochs = 0
    else:
        bad_epochs += 1

    print(
        f"Epoch {epoch:03d} | "
        f"lr {current_lr:.2e} | "
        f"lambda_vq {lambda_vq:.3f} | "
        f"train {train_stats['loss']:.6f} | "
        f"val {val_stats['loss']:.6f} | "
        f"recon {val_stats['recon']:.6f} | "
        f"vq {val_stats['vq']:.6f} | "
        f"skipped {train_stats['skipped']}"
    )
    run.log({"Epoch": epoch,
             "lr": current_lr,
             "lambda_vq": lambda_vq,
             "train": train_stats['loss'],
             "val": val_stats['loss'],
             "recon": val_stats['recon'],
             "vq": val_stats['vq'],
             "skipped": train_stats['skipped']})

    if bad_epochs >= patience:
        print("Early stopping.")
        break

model.load_state_dict(best_state)
model.to(device)

Epoch 001 | lr 1.00e-04 | lambda_vq 0.015 | train 0.002582 | val 0.002677 | recon 0.001562 | vq 0.074342 | skipped 0
Epoch 002 | lr 1.00e-04 | lambda_vq 0.030 | train 0.010071 | val 0.008010 | recon 0.000915 | vq 0.236511 | skipped 0
Epoch 003 | lr 1.00e-04 | lambda_vq 0.045 | train 0.006846 | val 0.006817 | recon 0.000813 | vq 0.133434 | skipped 0
Epoch 004 | lr 1.00e-04 | lambda_vq 0.060 | train 0.005808 | val 0.003009 | recon 0.000778 | vq 0.037179 | skipped 0
Epoch 005 | lr 1.00e-04 | lambda_vq 0.075 | train 0.002890 | val 0.003244 | recon 0.000723 | vq 0.033617 | skipped 0
Epoch 006 | lr 1.00e-04 | lambda_vq 0.090 | train 0.002932 | val 0.002544 | recon 0.000710 | vq 0.020380 | skipped 0
Epoch 007 | lr 1.00e-04 | lambda_vq 0.105 | train 0.002401 | val 0.002221 | recon 0.000700 | vq 0.014490 | skipped 0
Epoch 008 | lr 1.00e-04 | lambda_vq 0.120 | train 0.002213 | val 0.002203 | recon 0.000690 | vq 0.012606 | skipped 0
Epoch 009 | lr 1.00e-04 | lambda_vq 0.135 | train 0.002124 | val

VQVAE1DDenoiser(
  (enc): Encoder1D(
    (net): Sequential(
      (0): Conv1d(1, 32, kernel_size=(4,), stride=(2,), padding=(1,))
      (1): Tanh()
      (2): Conv1d(32, 32, kernel_size=(4,), stride=(2,), padding=(1,))
      (3): Tanh()
      (4): ResBlock1D(
        (net): Sequential(
          (0): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
          (1): ReLU(inplace=True)
          (2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
        )
      )
      (5): Conv1d(32, 512, kernel_size=(1,), stride=(1,))
    )
  )
  (vq): VectorQuantizer1D(
    (codebook): Embedding(32, 512)
  )
  (dec): Decoder1D(
    (net): Sequential(
      (0): Conv1d(512, 32, kernel_size=(1,), stride=(1,))
      (1): ResBlock1D(
        (net): Sequential(
          (0): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
          (1): ReLU(inplace=True)
          (2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
        )
      )
      (2): ConvTranspose

In [14]:
test_stats = run_epoch(test_loader, train=False, lambda_vq=0.25)
test_stats

{'loss': 0.0027160738536622375,
 'recon': 0.0006519987764477264,
 'vq': 0.00825630032340996,
 'skipped': 0}

In [15]:
model.eval()

batch = next(iter(test_loader))

x = batch["x"].to(device)
y = batch["y"].to(device)

with torch.no_grad():
    y_hat, vq_loss, codes = model(x)

for idx in range(5):
    noisy = x[idx, 0].cpu().numpy()
    clean = y[idx, 0].cpu().numpy()
    pred = y_hat[idx, 0].cpu().numpy()

    fig, ax = plt.subplots(figsize=(12, 4))

    ax.plot(clean, label="clean target", linewidth=2)
    ax.plot(noisy, label="noisy input", alpha=0.6)
    ax.plot(pred, label="VQ-VAE output", linewidth=2)

    ax.legend()
    ax.grid(True)
    fig.tight_layout()

    run.log({f"output_{idx}": fig})
    plt.close(fig)


/nix/store/zpnq8bysmwjszv9y6ypwvddm2vrpqkar-python3-3.10.5-env/lib/python3.10/site-packages/plotly/matplotlylib/renderer.py:612: UserWarning:

I found a path object that I don't think is part of a bar chart. Ignoring.

